# Create Like-for-Like Score Tables

Here i create both like-for-like tables and also a table which adjusts the min_token_size value to account for the fact on a problem-level it is categorical.

For example:
* We have a problem which has min_token_size of 1, 2, and 4
* We have other problems which have a min_token_size of 3
* Previously the mentioned problem would've been missed by min_token_size = 3 on the corpus level.
* Now we create min_token_size_adjusted which essentially duplicates the 4 row but allocates it to 3.

Then the lik-for-like uses this new column and will take the problems from each of these adjusted levels and gather all of the matching problems from the levels preceeding this to create a like-for-like grouping.

This is saved as two tables **raw_agg_adjusted_token_size_scores.rds** and **raw_agg_like_for_like_scores.rds** respectively.

In [93]:
import sys

import pandas as pd

from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from read_and_write_docs import write_rds

## Set Variables

In [94]:
models = ["gpt2"]

corpora = [
    "Wiki", "Enron", "Perverted Justice", "StackExchange", "ACL",
    "TripAdvisor", "The Apricity", "Koppel's Blogs", "The Telegraph",
    "Reddit", "Enron Cleaned"
]

data_types = ["training", "test"]

base_loc = "/Volumes/BCross/av_datasets_experiments/ngram_masking_logrpobs_n1"

file_name = "raw_agg_scores.xlsx"

adjusted_save_name = "raw_agg_adjusted_token_size_scores.rds"

like_for_like_save_name = "raw_agg_like_for_like_scores.rds"

## Function for adjusted_min_token_size

In [95]:
def get_distinct_problems(
    df,
    group_cols=['data_type', 'corpus', 'scoring_model', 'max_context_tokens', 'min_token_size']
):
    
    distinct_levels = (
        df
        .groupby(group_cols)
        .size()
        .reset_index(name='problem_count')
    )
    
    return distinct_levels
    

In [96]:
def add_adjusted_token_size(
    df,
    distinct_problems,
    base_cols=['data_type', 'corpus', 'scoring_model', 'max_context_tokens']
):
    """Calculates the adjusted token size e.g. if a problem have min_token_size 4
    but we have problems with min_token_size 3 then the problem with 4 will now also be included
    """
    # thresholds (the "new grouping") from distinct_levels
    levels = distinct_problems[base_cols + ['min_token_size']].rename(
        columns={'min_token_size': 'min_token_size_adjusted'}
    )
    
    tmp = df.merge(levels, on=base_cols, how='inner')

    # filter: df min_token_size >= threshold (adjusted bucket)
    tmp = tmp[tmp['min_token_size'] >= tmp['min_token_size_adjusted']]

    # partition by group + adjusted + problem, order by original min_token_size, take first
    tmp = tmp.sort_values(
        base_cols + ['min_token_size_adjusted', 'problem', 'min_token_size'],
        kind='mergesort'
    )

    tmp['row_number'] = (
        tmp.groupby(base_cols + ['min_token_size_adjusted', 'problem'])
        .cumcount()
        .add(1)
    )

    out = tmp[tmp['row_number'] == 1].drop(columns=['row_number']).copy()
    
    return out

In [97]:
def adjusted_token_size_pipeline(
    df,
    group_cols=['data_type', 'corpus', 'scoring_model', 'max_context_tokens', 'min_token_size'],
    base_cols=['data_type', 'corpus', 'scoring_model', 'max_context_tokens']
):
    
    distinct_levels = get_distinct_problems(df, group_cols=group_cols)
    
    adjusted_token_size_df = add_adjusted_token_size(df, distinct_levels, base_cols)
    
    return adjusted_token_size_df

## Like-for_Like Function

In [98]:
def get_like_for_like_problems(
    df,
    base_cols=['data_type', 'corpus', 'scoring_model', 'max_context_tokens']
):
    
    # Each (group, anchor_level) defines the problems that survive at that level
    anchors = (
        df[base_cols + ['problem', 'min_token_size_adjusted']]
        .drop_duplicates()
        .rename(columns={'min_token_size_adjusted': 'anchor_min_token_size_adjusted'})
    )

    # Attach each row in out to every anchor_level where that problem is in the anchor set
    out_likeforlike = df.merge(anchors, on=base_cols + ['problem'], how='inner')

    # Keep only levels up to (and including) the anchor level
    out_likeforlike = out_likeforlike[
        out_likeforlike['min_token_size_adjusted'] <= out_likeforlike['anchor_min_token_size_adjusted']
    ].copy()

    return out_likeforlike

## Run Code

In [99]:
for model_name in models:
    for data_type in data_types:
        for corpus in corpora:
            try:
                
                file_loc = f"{base_loc}/{data_type}/{corpus}/{model_name}/{file_name}"
                adjusted_save_loc = f"{base_loc}/{data_type}/{corpus}/{model_name}/{adjusted_save_name}"
                like_for_like_save_loc = f"{base_loc}/{data_type}/{corpus}/{model_name}/{like_for_like_save_name}"
                
                df = pd.read_excel(file_loc)
                
                adjusted_df  = adjusted_token_size_pipeline(df)
                
                write_rds(adjusted_df, adjusted_save_loc)
                
                like_for_like_df = get_like_for_like_problems(adjusted_df)
                
                write_rds(like_for_like_df, like_for_like_save_loc)
                print(f"Successful for model={model_name}, data_type={data_type}, corpus={corpus}")
            except Exception as e:
                print(f"Missing/failed for model={model_name}, data_type={data_type}, corpus={corpus}: {e}")

Successful for model=gpt2, data_type=training, corpus=Wiki
Missing/failed for model=gpt2, data_type=training, corpus=Enron: [Errno 2] No such file or directory: '/Volumes/BCross/av_datasets_experiments/ngram_masking_logrpobs_n1/training/Enron/gpt2/raw_agg_scores.xlsx'
Missing/failed for model=gpt2, data_type=training, corpus=Perverted Justice: [Errno 2] No such file or directory: '/Volumes/BCross/av_datasets_experiments/ngram_masking_logrpobs_n1/training/Perverted Justice/gpt2/raw_agg_scores.xlsx'
Missing/failed for model=gpt2, data_type=training, corpus=StackExchange: [Errno 2] No such file or directory: '/Volumes/BCross/av_datasets_experiments/ngram_masking_logrpobs_n1/training/StackExchange/gpt2/raw_agg_scores.xlsx'
Missing/failed for model=gpt2, data_type=training, corpus=ACL: [Errno 2] No such file or directory: '/Volumes/BCross/av_datasets_experiments/ngram_masking_logrpobs_n1/training/ACL/gpt2/raw_agg_scores.xlsx'
Missing/failed for model=gpt2, data_type=training, corpus=TripAdv